# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('data/day-of-week-not-scaled.csv')
df.head(5)

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [3]:
df_1 = pd.read_csv('data/dayofweek.csv')
df['dayofweek'] = df_1['dayofweek']
df.head(5)

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [4]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=21)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [5]:
params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}

In [6]:
search_svm = GridSearchCV(estimator=SVC(), param_grid=params, n_jobs=-1)
search_svm.fit(X_train, y_train)

GridSearchCV(estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid'],
                         'probability': [True], 'random_state': [21]})

In [7]:
search_svm.best_params_

{'C': 10,
 'class_weight': None,
 'gamma': 'auto',
 'kernel': 'rbf',
 'probability': True,
 'random_state': 21}

In [8]:
df_svm_analysis = pd.DataFrame({
    'params': search_svm.cv_results_['params'], 
    'score': search_svm.cv_results_['mean_test_score'],
    'rank': search_svm.cv_results_['rank_test_score']
}).sort_values('rank')

df_params = pd.DataFrame(df_svm_analysis['params'].tolist(), index=df_svm_analysis.index)
df_svm = pd.concat([df_params, df_svm_analysis[['score', 'rank']]], axis=1)

df_svm

,C,class_weight,gamma,kernel,probability,random_state,score,rank
70,10.0,None,auto,rbf,True,21,0.876109,1
64,10.0,balanced,auto,rbf,True,21,0.863500,2
58,5.0,None,auto,rbf,True,21,0.816018,3
52,5.0,balanced,auto,rbf,True,21,0.807865,4
63,10.0,balanced,auto,linear,True,21,0.721052,5
...,...,...,...,...,...,...,...,...
53,5.0,balanced,auto,sigmoid,True,21,0.129792,68
65,10.0,balanced,auto,sigmoid,True,21,0.115693,69
41,1.5,balanced,auto,sigmoid,True,21,0.079380,70
17,0.1,balanced,auto,sigmoid,True,21,0.062310,71


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [9]:
params = {
    'max_depth': [i for i in range(1, 50)],
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

In [10]:
search_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=params, n_jobs=-1)
search_tree.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...],
                         'random_state': [21]})

In [11]:
search_tree.best_params_

{'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': 23,
 'random_state': 21}

In [12]:
df_tree_analysis = pd.DataFrame({
    'params': search_tree.cv_results_['params'], 
    'score': search_tree.cv_results_['mean_test_score'],
    'rank': search_tree.cv_results_['rank_test_score']
}).sort_values('rank')

df_params = pd.DataFrame(df_tree_analysis['params'].tolist(), index=df_tree_analysis.index)
df_tree = pd.concat([df_params, df_tree_analysis[['score', 'rank']]], axis=1)

df_tree

,class_weight,criterion,max_depth,random_state,score,rank
97,balanced,gini,49,21,0.873859,1
95,balanced,gini,47,21,0.873859,1
94,balanced,gini,46,21,0.873859,1
71,balanced,gini,23,21,0.873859,1
72,balanced,gini,24,21,0.873859,1
...,...,...,...,...,...,...
51,balanced,gini,3,21,0.373906,192
147,None,gini,1,21,0.355330,193
98,None,entropy,1,21,0.355330,193
49,balanced,gini,1,21,0.286358,195


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [13]:
params = {
    'max_depth': [i for i in range(1, 50)],
    'n_estimators': [5, 10, 50, 100],
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

In [14]:
search_rf = GridSearchCV(estimator=RandomForestClassifier(), param_grid=params, n_jobs=-1)
search_rf.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...],
                         'n_estimators': [5, 10, 50, 100],
                         'random_state': [21]})

In [15]:
search_rf.best_params_

{'class_weight': None,
 'criterion': 'gini',
 'max_depth': 28,
 'n_estimators': 50,
 'random_state': 21}

In [16]:
df_rf_analysis = pd.DataFrame({
    'params': search_rf.cv_results_['params'], 
    'score': search_rf.cv_results_['mean_test_score'],
    'rank': search_rf.cv_results_['rank_test_score']
}).sort_values('rank')

df_params = pd.DataFrame(df_rf_analysis['params'].tolist(), index=df_rf_analysis.index)
df_rf = pd.concat([df_params, df_rf_analysis[['score', 'rank']]], axis=1)

df_rf

,class_weight,criterion,max_depth,n_estimators,random_state,score,rank
698,None,gini,28,50,21,0.904290,1
711,None,gini,31,100,21,0.904287,2
374,balanced,gini,45,50,21,0.903549,3
390,balanced,gini,49,50,21,0.903549,3
386,balanced,gini,48,50,21,0.903549,3
...,...,...,...,...,...,...,...
392,None,entropy,1,5,21,0.353832,780
4,balanced,entropy,2,5,21,0.353110,781
200,balanced,gini,2,5,21,0.346419,782
196,balanced,gini,1,5,21,0.283390,783


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [17]:
params = {
    'max_depth': [i for i in range(1, 50)],
    'n_estimators': [5, 10, 50, 100],
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

In [18]:
results = []

for depth in tqdm(range(1, 50)):
    for estimators in (5, 10, 50, 100):
        for weight in ('balanced', None):
            for criterion in ('entropy', 'gini'):
                model = RandomForestClassifier(max_depth=depth, n_estimators=estimators, 
                                               class_weight=weight, criterion=criterion, random_state=21)
                scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
                mean_accuracy = scores.mean()
                std_accuracy = scores.std()
                
                results.append({
                    'max_depth': depth,
                    'n_estimators': estimators,
                    'class_weight': weight,
                    'criterion': criterion,
                    'accuracy_mean': mean_accuracy,
                    'accuracy_std': std_accuracy
                })
                
df_results = pd.DataFrame(results).sort_values('accuracy_mean', ascending=False)             

  0%|          | 0/49 [00:00<?, ?it/s]

In [19]:
df_results

,max_depth,n_estimators,class_weight,criterion,accuracy_mean,accuracy_std
443,28,50,None,gini,0.904290,0.010961
495,31,100,None,gini,0.904287,0.015204
569,36,50,balanced,gini,0.903549,0.012503
617,39,50,balanced,gini,0.903549,0.012503
713,45,50,balanced,gini,0.903549,0.012503
...,...,...,...,...,...,...
2,1,5,None,entropy,0.353832,0.016467
16,2,5,balanced,entropy,0.353110,0.021165
17,2,5,balanced,gini,0.346419,0.029749
1,1,5,balanced,gini,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [20]:
final_model = RandomForestClassifier(max_depth=28, n_estimators=50, criterion='gini', random_state=21)
final_model.fit(X_train, y_train)
predictions = final_model.predict(X_test)

final_accuracy = accuracy_score(y_test, predictions)
print(f'Final accuracy: {final_accuracy:.4f}')

Final accuracy: 0.9290
